# scDesignPop Experiment

This notebook is designed to mimic the scDesignPop Quick Start. We use a `NegBinEQTLCopula` simulator class, which a negative binomial model where each gene can be influenced by its own set of SNPs and each donor can have a random intercept. Specifically, for gene $j$, we model,

$$
\begin{align*}
Y_{ij}\mid u_{k(i)j}\;&\sim\;\mathrm{NB}\!\left(\mu_{ij},\,r_j\right) \\
\log\mu_{ij}&=W_i^{\top}\theta_j+u_{k(i)j}+D_{k(i)j}\left(\beta_j+V_i^{\top}\zeta_j\right) \\
u_{kj}&\sim N\!\left(0,\sigma_j^{2}\right),
\end{align*}
$$

where $D_{k(i)j}$ are the genotype effects, $\zeta_{j}$ accounts for interactions (e.g., cell type-specific effects), and $u_{kj}$ is donor $k$'s random intercept. We use a copula to relate the genes.

The anndata objects we're working with have this structure:

| source | interpretation |
|---|---|
| `X` | counts, cells x genes |
| `obs` | `indiv` (the donor) and `cell_type` |
| `var["snp"]` | the cis SNP regulating each gene |
| `uns["dosage"]` | donors x SNPs, entries in $\{0, 1, 2\}$ |

We load a version of the quickstart data with five of the most highly varying genes omitted. You can try re-running the vignette with those as well, but we noticed that for those genes, the negative binomial is a very poor fit. For example, for gene ENSG00000124491, the mean is 4.5e11, even though the 95% of the values are 0! You can try that version by uncommenting the currently commented line. For those five genes we severely underestimate the mean because we seem not to be able to capture the outliers.

In [1]:
import numpy as np
import pandas as pd
import time
from anndata import read_h5ad
from scdesigner.simulators import NegBinEQTLCopula

DATA = "data/eqtl_bench"
adata = read_h5ad(f"{DATA}/quickstart977.h5ad")
#adata = read_h5ad(f"{DATA}/quickstart982.h5ad")
dosage = adata.uns["dosage"]     # donors x SNPs, in {0, 1, 2}
snp_map = {gene: [snp] for gene, snp in adata.var["snp"].items()}
adata

AnnData object with n_obs × n_vars = 7998 × 977
    obs: 'indiv', 'cell_type'
    var: 'snp'
    uns: 'dosage'

## Model Estimation

Here are arguments for initializing the simulator. 

| argument | meaning |
|---|---|
| `mean_formula` | cell-level covariates for $\log\mu$ |
| `interaction_formula` | genotype effect modifiers. E.g., `~ 0 + C(cell_type)` defines cell type specific eQTLs and `~ bs(pseudotime, df=5)` defines dynamic eQTL |
| `donor_col` | the column of `adata.obs` holding donor IDs |
| `dosage` | donors x SNPs, entries in $\{0, 1, 2\}$. |
| `snp_map` | gene to the SNPs regulating it |

In principle, we could add covariates when modeling the dispersion and copula correlations, but we've left that out of this notebook. We're running on an laptop with an MPS GPU, but you can alternatively set `device="cpu"`.

In [ ]:
sim = NegBinEQTLCopula(
    mean_formula="~ C(cell_type)",
    interaction_formula="~ 0 + C(cell_type)",
    donor_col="indiv",
    dosage=dosage,
    snp_map=snp_map,
    device="mps",
)

start_time = time.perf_counter()
sim.fit(adata, verbose=False)
print(f"Estimation time: {time.perf_counter() - start_time:.2f} seconds")

Like all our scdesigner models, the fitted parameters can be found under `sim.parameters`. A parameter of particular interest in this model is `sigma2`, which parameterizes the between-donor variance. This is important for simulating new donors.  

In [ ]:
sigma2 = sim.marginal.sigma2.cpu().numpy()
print(f"sigma^2 over {len(sigma2)} genes: median {np.median(sigma2):.3f}, "
      f"IQR {np.percentile(sigma2, 25):.3f}-{np.percentile(sigma2, 75):.3f}")

## Simulate

There are two ways to simulate new samples. We can simulate new cells using the existing donors. This is conditioning on the fitted `U`. Alternatively, we can simulate a new cohort by marginalizing over `U` with the estimated $\sigma_j^2$.  This is what the `simulated_donor_effects` register does.

In [ ]:
conditional = sim.sample()

with sim.marginal.simulated_donor_effects():
    marginal = sim.sample()

donor_codes = adata.obs["indiv"].map(sim.marginal.donor_to_code).to_numpy()
n_donors = dosage.shape[0]

## Diagnostics

We check the model quality using our accompanying `scdiagnostics` package. We first look at gene-level statistics after first `log1p`-normalizing. 

In [ ]:
import altair as alt
from anndata import AnnData
import scdiagnostics as scd

alt.renderers.enable('png')
alt.data_transformers.disable_max_rows()

For each of these plots, there is one point for every gene. We also color for whether the gene has any donors with no counts for that gene at all. These tend to be more difficult to estimate (it seems related to the quality of the Laplace approximation, we have other notes exploring this).

In [ ]:
empty_donor = (scd.data.pseudobulk(adata.X, donor_codes, n_donors) == 0).any(axis=0)
split = np.where(empty_donor, "a donor with no counts", "every donor observed")
print(f"{empty_donor.sum()} of {len(empty_donor)} genes have a donor with zero counts")

zero_fraction = lambda a: (a.X == 0).mean(axis=0)

alt.hconcat(
    scd.compare_means(adata, conditional, color=split).properties(title="mean, log1p"),
    scd.compare_variances(adata, conditional, color=split).properties(title="variance, log1p"),
    scd.compare_summary(adata, conditional, zero_fraction, color=split).properties(title="fraction of zeros"),
)

Here are some quantitative summaries of the previous plots.

In [ ]:
statistics = {
    "mean": lambda X: np.log1p(X).mean(axis=0),
    "variance": lambda X: np.var(np.log1p(X), axis=0),
    "fraction of zeros": lambda X: (X == 0).mean(axis=0),
}

pd.DataFrame([
    {"quantity": name, "median real": np.median(f(adata.X)),
     "median simulated": np.median(f(conditional.X)),
     "correlation": np.corrcoef(f(adata.X), f(conditional.X))[0, 1]}
    for name, f in statistics.items()
]).round(4)

Let's ask the same question again. Let us ask the same question, restricted within specific cell types. This is useful for seeing whether any particular cells are more difficult to estimate than others (if that were the case, that would mean the interaction terms weren't working well).

In [ ]:
def within(cell_type):
    keep = adata.obs["cell_type"] == cell_type
    return scd.compare_means(adata[keep], conditional[keep]).properties(
        title=f"{cell_type}  (n={int(keep.sum()):,} cells)")

alt.hconcat(*[within(t) for t in adata.obs["cell_type"].cat.categories])

Next, we study the quality of the estimated between-donor variance using pseuobulk summarized data. The plots below show both the conditional and the marginalized versions for comparison, but only the marginalized version actually draws a new cohort. We seem to sometimes underestimate the between-donor variance for the very high variance genes, but this also appears even in the conditional case. There seem to be donors with a very high expression for some genes, which have high influence on the real data variants. 

In [ ]:
def donor_pseudobulk(a):
    """Donors x genes mean expression."""
    return AnnData(scd.data.pseudobulk(a.X, donor_codes, n_donors), var=a.var)

real_pb, cond_pb, marg_pb = (donor_pseudobulk(a) for a in (adata, conditional, marginal))

alt.hconcat(
    scd.compare_variances(real_pb, cond_pb).properties(title="conditional — reuse fitted U"),
    scd.compare_variances(real_pb, marg_pb).properties(title="marginal — redraw u ~ N(0, sigma^2)"),
)

Here are some quantitative summaries. 

In [ ]:
real_variance = np.nanvar(np.log1p(real_pb.X), axis=0)

rows = []
for regime, pb in [("conditional", cond_pb), ("marginal", marg_pb)]:
    simulated_variance = np.nanvar(np.log1p(pb.X), axis=0)
    keep = (real_variance > 0) & (simulated_variance > 0)
    rows.append({"regime": regime, "n_genes": int(keep.sum()),
                 "median simulated / real": np.median(simulated_variance[keep] / real_variance[keep])})

pd.DataFrame(rows).round(3)

Finally, we compare the gene-gene correlations. This evaluates the quality of the copula.

In [ ]:
top = np.argsort(-np.var(adata.X, axis=0))[:200]
upper = np.triu_indices(len(top), k=1)

corr = pd.DataFrame({
    name: np.corrcoef(np.log1p(X[:, top]), rowvar=False)[upper]
    for name, X in [("real", adata.X), ("simulated", conditional.X)]
})

(alt.Chart(corr.melt(var_name="dataset", value_name="correlation"))
    .transform_density("correlation", groupby=["dataset"],
                       as_=["correlation", "density"])
    .mark_line()
    .encode(
        x=alt.X("correlation:Q", title="pairwise correlation"),
        y=alt.Y("density:Q", stack=None),
        color=alt.Color("dataset:N", title=None),
))

In [ ]:
alt.Chart(corr).mark_circle(size=16, opacity=0.15).encode(
    x=alt.X("real:Q"), y=alt.Y("simulated:Q"))